# Cross-entropy loss

_Deep Learning_

**Measures how wrong a yes/no prediction is. Confident and wrong hurts the most.**

A loss is a number that says how wrong a prediction is. Smaller is better.

     Cross-entropy is the loss we use when the answer is yes/no (1 or 0).

---

This notebook is a practice scaffold for the **Cross-entropy loss** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Set up the loss and a batch of predictions

`BCEWithLogitsLoss` combines a sigmoid and binary cross-entropy in one numerically stable step, so we feed it the raw scores (**logits**) straight from the last layer rather than probabilities. We make three example predictions and their true yes/no labels (`1` = yes, `0` = no).

In [ ]:
import torch
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss()        # sigmoid + cross-entropy in one

logits = torch.tensor([2.2, -1.5, 0.3])  # raw scores from the last layer
y = torch.tensor([1.0, 0.0, 1.0])         # true labels: 1 = yes, 0 = no

### Step 2 — Compute the loss

The loss function turns each logit into a probability and compares it to the true label, then averages the cost across the three examples. A single number comes out: lower means the predictions were closer to the truth.

In [ ]:
loss = loss_fn(logits, y)               # average loss over the 3 examples
print("loss:", round(loss.item(), 4))

### Step 3 — See the probabilities behind the loss

Passing the logits through a sigmoid shows the model's actual predicted probability for each example. The third example (logit `0.3`, true label `1`) is only weakly confident, which is where most of the loss comes from — cross-entropy punishes predictions that are confident **and** wrong the hardest.

In [ ]:
# confident-and-wrong costs the most:
probs = torch.sigmoid(logits)
print("p:", probs.round(decimals=3).tolist())

## Visualize the data & results

_For a real digit the classifier scored as a 5 with probability 0.69, how big is the cross-entropy loss?_

### Step 1 — Train a digit classifier

We fit a small neural network on the digit images (pixel values scaled to 0–1). It learns to output a probability for each of the ten digit classes via a softmax, which we'll inspect next.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0

net = MLPClassifier(
    hidden_layer_sizes=(16,),
    max_iter=300,
    random_state=0,
    alpha=1e-3,
).fit(X, digits.target)

### Step 2 — Find a real, not-overconfident prediction

`predict_proba` gives the softmax probability the model assigned to every class. We pull out the probability it gave to the *correct* digit for each image, then pick one example whose confidence is middling (between 0.35 and 0.8). For that example the cross-entropy loss is simply `-log(p)` of the true-class probability.

In [ ]:
probs = net.predict_proba(X)            # softmax over the 10 digit classes
conf = probs[np.arange(len(X)), digits.target]

# pick the first sample that is correct but not overconfident
i = np.where((conf > 0.35) & (conf < 0.8))[0][0]

p_true = probs[i, digits.target[i]]     # real probability for the true digit (0.693)
loss = -np.log(p_true)                  # real cross-entropy = 0.367

### Step 3 — Plot the loss curve and mark the real point

We draw the cross-entropy curve `-log(p)` across all possible probabilities for the true class, then mark where our real digit landed. The curve shoots up as `p` approaches 0 (confidently wrong) and falls to 0 as `p` approaches 1 (confidently right) — showing exactly how the loss rewards correct, confident predictions.

In [ ]:
p = np.linspace(0.02, 1.0, 30)

plt.plot(p, -np.log(p), color="#ff7b72", label="loss")
plt.scatter([p_true], [loss], color="#7ee787", zorder=3)
plt.title("Cross-entropy = -log(p) for the true class; real digit scored p=0.69")
plt.xlabel("predicted probability for true class")
plt.ylabel("loss")
plt.legend()
plt.show()